# KubeCon India 2026 — Environment Setup (AWS / ACK track)

The notebook form of [`init-with-ack.sh`](init-with-ack.sh) — Track 2: a local cluster
+ KubeVela + the **AWS Controllers for Kubernetes (ACK)** S3 controller as the
cloud-resource orchestrator. Reuses the repo's [`../../../scripts/`](../../../scripts/)
helpers, so the notebook and the script never drift.

## Prerequisites
- `k3d`, `kubectl`, `helm`, `vela`, `docker`, `python3`
- AWS credentials in `aws-setup/.env.aws` (template written on first run if missing)

## Steps
0. Check prerequisites
1. Python venv + load configuration
2. Create the k3d cluster + local registry
3. Establish AWS connectivity (`.env.aws`)
4. Install the ACK S3 controller (Helm, OCI chart)
5. Install the KubeVela control plane (+ VelaUX)

> After this, run [`01_setup-with-ack.ipynb`](01_setup-with-ack.ipynb).


## Step 0: Prerequisites

In [ ]:
# Prerequisites check
import shutil, os, sys, subprocess

print("=== Checking prerequisites ===\n")
tools = ["k3d", "kubectl", "helm", "vela", "docker", "python3"]
missing = [t for t in tools if shutil.which(t) is None]
for t in tools:
    print(f"  {'OK ' if shutil.which(t) else 'MISSING'}  {t}")
print("\nconfig.yaml:", "OK" if os.path.exists("config.yaml") else "MISSING")
try:
    import yaml  # noqa: F401
    print("PyYAML:     OK")
except ImportError:
    print("PyYAML:     installing into the kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    print("PyYAML:     OK")
if missing:
    print("\n*** Install the missing tools before continuing:")
    print("    brew install k3d kubectl helm docker python@3.12\ncurl -fsSl https://kubevela.io/script/install.sh | bash   # vela")
elif not os.path.exists("config.yaml"):
    print("\n*** config.yaml is missing from this folder.")
else:
    print("\nAll prerequisites satisfied.")


## Step 1: Python venv + load configuration

Create/activate the demo-local venv (installs PyYAML), then run `load_config` to parse
this folder's `config.yaml`, export the cluster vars, and write `.env.sh` (sourced by
later cells).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/setup-venv.sh"
source "$REPO_ROOT/scripts/load-config.sh"

setup_venv "$PWD/.venv" "$REPO_ROOT/scripts/requirements.txt"
load_config "$PWD/config.yaml"
echo
echo "Config: CLUSTER_NAME=$CLUSTER_NAME  API_PORT=$API_PORT  HTTP_PORT=$HTTP_PORT"


## Step 2: Create the k3d cluster + local registry

`create_cluster` recreates the cluster wired to a local registry, switches the kubectl
context, and verifies access (uses the vars from Step 1).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/create-cluster.sh"

create_cluster


## Step 3: Establish AWS connectivity

`load_aws_env` sources `aws-setup/.env.aws` and exports `AWS_*`. If missing it writes a
template and stops — fill it in, then re-run.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-aws-env.sh"

load_aws_env "$PWD/.env.aws"


## Step 4: Install the ACK S3 controller

`install_ack_controller` Helm-installs the ACK S3 controller from its public-ECR OCI
chart (no `helm repo add`), creating the `aws-credentials` secret in the `ack-system`
namespace from the loaded `.env.aws` and pointing the chart at it. This cell re-loads
`.env.aws` first (fresh shell).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-aws-env.sh"
source "$REPO_ROOT/scripts/create-aws-secret.sh"
source "$REPO_ROOT/scripts/install-ack.sh"

load_aws_env "$PWD/.env.aws"
install_ack_controller          # service s3, ns ack-system (defaults)


## Step 5: Install the KubeVela control plane

`install_kubevela --velaux` installs `vela-core`, enables VelaUX, and port-forwards it
to http://localhost:8000.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/install-kubevela.sh"

install_kubevela --velaux


## Setup complete

The ACK track is bootstrapped: k3d cluster + registry, the ACK S3 controller, and the
KubeVela control plane (+ VelaUX). Next: [`01_setup-with-ack.ipynb`](01_setup-with-ack.ipynb).
